In [1]:
import pandas as pd

results = pd.read_csv("../output/open_firstImpulse0.15.csv")

print(f"Rows: {len(results):,}")
print("\nColumns:")
print(results.columns.tolist())

results.head()

Rows: 4,437

Columns:
['Date', 'FirstImpulse', 'ImpulseMFE', 'ImpulseMAE']


,Date,FirstImpulse,ImpulseMFE,ImpulseMAE
0,2009-01-02,Up,58.695623,1.164406
1,2009-01-05,Down,4.644134,30.516369
2,2009-01-06,Up,10.509752,14.480357
3,2009-01-07,Down,23.950238,10.338516
4,2009-01-08,Down,6.399045,27.017961


In [2]:
print("Missing values:")
display(results.isna().sum())

print("\nFirstImpulse distibution:")
display(results["FirstImpulse"].value_counts(dropna=False).sort_index())

print("\nSummary statistics:")
display(results.describe(include="all"))

Missing values:


Date            0
FirstImpulse    0
ImpulseMFE      0
ImpulseMAE      0
dtype: int64


FirstImpulse distibution:


FirstImpulse
Down    2220
Up      2217
Name: count, dtype: int64


Summary statistics:


,Date,FirstImpulse,ImpulseMFE,ImpulseMAE
count,4437,4437,4437.000000,4437.000000
unique,4437,2,NaN,NaN
top,2009-01-02,Down,NaN,NaN
freq,1,2220,NaN,NaN
mean,NaN,NaN,69.941441,65.145985
std,NaN,NaN,103.221065,97.926701
min,NaN,NaN,0.004248,0.117976
25%,NaN,NaN,12.285426,11.975187
50%,NaN,NaN,30.250694,28.364572
75%,NaN,NaN,84.912167,75.927818


In [3]:
sample = results.sample(5, random_state=42).sort_values("Date")

display(sample)

,Date,FirstImpulse,ImpulseMFE,ImpulseMAE
376,2010-06-23,Down,20.293283,16.801696
468,2010-11-01,Up,12.306905,24.544868
1361,2014-05-05,Down,0.973708,58.516506
2792,2019-12-19,Up,40.050248,10.061551
3682,2023-07-05,Up,120.253691,37.600495


In [4]:
import psycopg
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM candles
WHERE timestamp >= %(date)s::date
AND timestamp < (%(date)s::date + INTERVAL '1 day')
ORDER BY timestamp;
"""

with psycopg.connect("dbname=dailyedge_development") as connection:
    minute = pd.read_sql(query, connection, params={"date": "2010-06-23"})

print(len(minute))
minute.head()

1265


/tmp/ipykernel_20911/2660316789.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  minute = pd.read_sql(query, connection, params={"date": "2010-06-23"})


,timestamp,open,high,low,close,volume
0,2010-06-23 00:00:00,2199.995121,2199.995121,2199.995121,2199.995121,1
1,2010-06-23 00:02:00,2199.995121,2200.579294,2199.995121,2200.579294,3
2,2010-06-23 00:04:00,2200.287207,2200.287207,2200.287207,2200.287207,1
3,2010-06-23 00:10:00,2199.703034,2199.703034,2199.703034,2199.703034,1
4,2010-06-23 00:12:00,2200.287207,2200.287207,2200.287207,2200.287207,1


In [5]:
minute["timestamp"] = pd.to_datetime(minute["timestamp"])

rth = minute[
    (minute["timestamp"].dt.time >= pd.Timestamp("09:30").time()) &
    (minute["timestamp"].dt.time < pd.Timestamp("16:00").time())
].copy()

print(f"RTH rows: {len(rth)}")

rth.head()

RTH rows: 373


,timestamp,open,high,low,close,volume
538,2010-06-23 09:30:00,2171.370649,2174.875686,2170.494390,2173.999427,1329
539,2010-06-23 09:31:00,2173.707341,2174.583600,2173.123168,2174.291514,560
540,2010-06-23 09:32:00,2174.583600,2176.628205,2174.291514,2175.751946,1431
541,2010-06-23 09:33:00,2175.751946,2179.841156,2175.459859,2178.088637,1474
542,2010-06-23 09:34:00,2178.088637,2181.009502,2177.796551,2180.425329,927


In [6]:
session = results.loc[results["Date"] == "2010-06-23"]

display(session)

,Date,FirstImpulse,ImpulseMFE,ImpulseMAE
376,2010-06-23,Down,20.293283,16.801696


In [7]:
daily = pd.read_csv("../output/daily_stats.csv")

daily["Date"] = pd.to_datetime(daily["Date"]).dt.strftime("%Y-%m-%d")

daily.loc[daily["Date"] == "2010-06-23"]

,Date,Open,High,Low,Close,Direction,Extension,Counter
473,2010-06-23,2199.995121,2208.173541,2170.49439,2188.311663,Bear,29.500731,8.17842


In [8]:
open_price = rth.iloc[0]["open"]

print(open_price)

2171.370649


In [9]:
# Convert Date column
results["Date"] = pd.to_datetime(results["Date"])

# Keep only last year
latest = results["Date"].max()
one_year = latest - pd.DateOffset(years=1)

df = results[results["Date"] >= one_year].copy()

print(f"Sessions: {len(df)}")

Sessions: 254


In [10]:
summary = (
    df.groupby("FirstImpulse")
      .agg(
          Sessions=("FirstImpulse", "count"),
          Median_MFE=("ImpulseMFE", "median"),
          Mean_MFE=("ImpulseMFE", "mean"),
          Median_MAE=("ImpulseMAE", "median"),
          Mean_MAE=("ImpulseMAE", "mean"),
      )
)

summary.loc["Total"] = {
    "Sessions": len(df),
    "Median_MFE": df["ImpulseMFE"].median(),
    "Mean_MFE": df["ImpulseMFE"].mean(),
    "Median_MAE": df["ImpulseMAE"].median(),
    "Mean_MAE": df["ImpulseMAE"].mean(),
}

summary.round(2)

,Sessions,Median_MFE,Mean_MFE,Median_MAE,Mean_MAE
FirstImpulse,,,,,
Down,135,145.39,190.64,97.03,147.54
Up,119,139.76,158.59,99.64,150.92
Total,254,140.76,175.63,99.13,149.12


In [11]:
targets = range(25, 225, 25)

rows = []

for target in targets:
    winners = df[df["ImpulseMFE"] >= target]

    rows.append({
        "Extension": target,
        "Continuation %": round(100 * len(winners) / len(df), 1),
        "Median MAE": winners["ImpulseMAE"].median(),
        "Median MFE": winners["ImpulseMFE"].median(),
        "Count": len(winners)
    })

continuation = pd.DataFrame(rows)

continuation

,Extension,Continuation %,Median MAE,Median MFE,Count
0,25,90.9,92.516519,155.010363,231
1,50,81.5,76.646447,166.155686,207
2,75,72.4,68.507826,176.038022,184
3,100,61.4,65.056400,206.607806,156
4,125,55.5,64.453171,219.521620,141
5,150,47.2,64.659225,241.874023,120
6,175,38.2,63.561917,267.424056,97
7,200,31.9,64.453171,295.539364,81
